# MACE training with different values of the --correlation parameter

Given how MACE is built, differently from previous implementations of similar models, the number of atoms considered from a single interaction does not result in an exponential scaling in the computational complexity. As a result, in this section we analyze how different number of contemporary interaction change the learning and the physical results.

## --correlation=2

In [ ]:
import os
import sys
import numpy as np
from ase.io import read, write
from google.colab import drive

from mace.cli.run_train import main

# Mount Google Drive to access the MDProject folder
drive.mount('/content/drive')

# Define file paths
outcar_file = "/content/drive/MyDrive/MDProject/OUTCAR"

data_dir = "/content/drive/MyDrive/MDProject/new_data/1.NVT_300/A.1_10K/corr2"
my_sim_checkpoints_dir = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/corr2true"
my_sim_results_dir = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/results/corr2true"

os.makedirs(data_dir, exist_ok=True)

train_file = f"{data_dir}/train.extxyz"
valid_file = f"{data_dir}/valid.extxyz"
test_file = f"{data_dir}/test.extxyz"

frames = read(outcar_file, index="6840::10", format="vasp-out")
print(f"Extracted {len(frames)} distinct frames.")

# FIX ASE KEYS
print("Renaming ASE internal keys...")
for frame in frames:
    if 'energy' in frame.info:
        frame.info['REF_energy'] = frame.info.pop('energy')
    if 'stress' in frame.info:
        frame.info['REF_stress'] = frame.info.pop('stress')
    if 'forces' in frame.arrays:
        frame.arrays['REF_forces'] = frame.arrays.pop('forces')

# SPLIT: 80% Train, 10% Valid, 10% Test
n_total = len(frames)
n_train = int(0.8 * n_total)
n_valid = int(0.1 * n_total)

train_frames = frames[:n_train]
valid_frames = frames[n_train : n_train + n_valid]
test_frames = frames[n_train + n_valid:]

# SAVE
write(train_file, train_frames, format="extxyz")
write(valid_file, valid_frames, format="extxyz")
write(test_file, test_frames, format="extxyz")

print(f"Saved {len(train_frames)} train, {len(valid_frames)} valid, and {len(test_frames)} test frames.")

my_atomic_numbers = "[1, 6, 7, 50, 53]"

custom_args = [
    "mace_run_training",
    "--name=model_3000_pts_corr2",
    f"--train_file={train_file}",
    f"--valid_file={valid_file}",

    "--energy_key=energy",
    "--forces_key=forces",
    "--E0s=average",

    "--stress_weight=1.0",
    "--compute_stress=True",

    "--max_num_epochs=200",
    "--patience=10",
    "--batch_size=5",
    "--valid_batch_size=5",
    "--device=cuda",
    "--default_dtype=float32",

    "--r_max=5.0",
    "--correlation=2",

    f"--checkpoints_dir={my_sim_checkpoints_dir}",
    f"--results_dir={my_sim_results_dir}",
    "--keep_checkpoints",

    "--num_channels=64",
    "--max_L=0",
    "--seed=1",


    f"--atomic_numbers={my_atomic_numbers}",
    "--num_samples_pt=1000",
    "--lr=0.0001",

    "--energy_weight=1.0",
    "--forces_weight=100.0",

    "--optimizer=adamw",
    "--amsgrad",
    "--weight_decay=5e-7",

    "--ema",
    "--ema_decay=0.99",

    "--swa",
    "--start_swa=30",

    "--swa_forces_weight=100.0",
    "--swa_stress_weight=1.0",
    "--swa_energy_weight=1.0",

    "--scaling=rms_forces_scaling",
    "--restart_latest"
]

sys.argv = custom_args

if __name__ == "__main__":
    try:
        main()
        print("Training completed!")
    except Exception as e:
        print(f"Error: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading and subsampling OUTCAR...
Extracted 316 distinct frames.
Renaming ASE internal keys...


INFO:root:===========VERIFYING SETTINGS===========


Saved 252 train, 31 valid, and 33 test frames.
2026-09-21 12:36:11.632 INFO: ===========VERIFYING SETTINGS===========
2026-09-21 12:36:11.632 INFO: ===========VERIFYING SETTINGS===========
2026-09-21 12:36:11.632 INFO: ===========VERIFYING SETTINGS===========


INFO:root:Stage Two is activated as start_stage_two was defined


2026-09-21 12:36:11.636 INFO: Stage Two is activated as start_stage_two was defined
2026-09-21 12:36:11.636 INFO: Stage Two is activated as start_stage_two was defined
2026-09-21 12:36:11.636 INFO: Stage Two is activated as start_stage_two was defined


...

2026-09-21 12:44:56.265 INFO: Evaluating valid_Default ...
2026-09-21 12:44:56.265 INFO: Evaluating valid_Default ...
2026-09-21 12:44:56.265 INFO: Evaluating valid_Default ...


INFO:root:Error-table on TRAIN and VALID:
+---------------+---------------------+------------------+-------------------+
|  config_type  | RMSE E / meV / atom | RMSE F / meV / A | relative F RMSE % |
+---------------+---------------------+------------------+-------------------+
| train_Default |            2.6      |         48.7     |         14.49     |
| valid_Default |            2.5      |         56.6     |         16.34     |
+---------------+---------------------+------------------+-------------------+


2026-09-21 12:44:56.533 INFO: Error-table on TRAIN and VALID:
+---------------+---------------------+------------------+-------------------+
|  config_type  | RMSE E / meV / atom | RMSE F / meV / A | relative F RMSE % |
+---------------+---------------------+------------------+-------------------+
| train_Default |            2.6      |         48.7     |         14.49     |
| valid_Default |            2.5      |         56.6     |         16.34     |
+---------------+---------------------+------------------+-------------------+
2026-09-21 12:44:56.533 INFO: Error-table on TRAIN and VALID:
+---------------+---------------------+------------------+-------------------+
|  config_type  | RMSE E / meV / atom | RMSE F / meV / A | relative F RMSE % |
+---------------+---------------------+------------------+-------------------+
| train_Default |            2.6      |         48.7     |         14.49     |
| valid_Default |            2.5      |         56.6     |         16.34     |
+------

DEBUG:root:Running inference on train_Default dataset
DEBUG:root:Running inference on valid_Default dataset
INFO:root:Done


2026-09-21 12:45:01.103 INFO: Done
2026-09-21 12:45:01.103 INFO: Done
2026-09-21 12:45:01.103 INFO: Done
Training completed!


Simulating trajectory with ase

In [ ]:
import os
import time
from ase.io import read, write
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from mace.calculators import MACECalculator
from google.colab import drive

drive.mount('/content/drive')

model_path = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/corr2true/model_3000_pts_corr2_run-1_stagetwo.model"
init_structure_path = "/content/drive/MyDrive/MDProject/new_data/1.NVT_300/A.1_10K/Smart_FPS/valid_fps.extxyz"
output_traj_path = "/content/drive/MyDrive/MDProject/results/md_trajectory_corr2_2k.extxyz"


atoms = read(init_structure_path, index=0)

#  Initialize the MACE Calculator with your compiled model
print(f"Loading compiled MACE model from: {model_path}...")
calculator = MACECalculator(
    model_paths=[model_path],
    device='cuda',
    default_dtype='float32'
)
atoms.calc = calculator

#  Set up the MD Simulation (NVT Ensemble at 300K)
temperature_K = 300
timestep_fs = 1.0
friction_coeff = 0.001

print(f"Initializing velocities at {temperature_K} K...")
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)

dyn = Langevin(
    atoms,
    timestep_fs * units.fs,
    temperature_K=temperature_K,
    friction=friction_coeff
)

# 6. Define an output function to save frames
def write_frame():
    epot = atoms.get_potential_energy()
    ekin = atoms.get_kinetic_energy()
    temp = ekin / (1.5 * units.kB * len(atoms))

    # Append to the .extxyz trajectory file
    write(output_traj_path, atoms, append=True)

    # Print status every 10 steps
    step = dyn.get_number_of_steps()
    print(f"Step: {step:>4} | Temp: {temp:>6.1f} K | Epot: {epot:+.3f} eV | Total E: {epot+ekin:+.3f} eV")

# Clear old trajectory file if it exists
if os.path.exists(output_traj_path):
    os.remove(output_traj_path)

# Attach the logger to run every 10 steps
dyn.attach(write_frame, interval=1)

#  Run the Simulation
num_steps = 2000
print(f"\nStarting MD simulation for {num_steps} steps...")
start_time = time.time()

dyn.run(num_steps)

print(f"\nMD simulation finished in {time.time() - start_time:.2f} seconds!")
print(f"Trajectory safely saved to: {output_traj_path}")

/usr/local/lib/python3.13/dist-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Mounted at /content/drive
Loading initial structure...
Loading compiled MACE model from: /content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/corr2true/model_3000_pts_corr2_run-1_stagetwo.model...


/usr/local/lib/python3.13/dist-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Initializing velocities at 300 K...

Starting MD simulation for 2000 steps...


/tmp/ipykernel_565/4053206790.py:38: DeprecationWarning: Use thermalize_momenta
  MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)
/usr/local/lib/python3.13/dist-packages/ase/md/langevin.py:102: FutureWarning: The implementation of `fixcm=True` in `Langevin` does not strictly sample the correct NVT distributions. The deviations are typically small for large systems but can be more pronounced for small systems. Use `fixcm=False` together with `ase.constraints.FixCom`. `fixcm` is deprecated since ASE 3.28.0 and will be removed in a future release.
  warnings.warn(msg, FutureWarning)


Step:    0 | Temp:  269.3 K | Epot: -479.262 eV | Total E: -475.780 eV
Step:    1 | Temp:  250.5 K | Epot: -479.002 eV | Total E: -475.764 eV
Step:    2 | Temp:  213.1 K | Epot: -478.484 eV | Total E: -475.729 eV
Step:    3 | Temp:  188.8 K | Epot: -478.155 eV | Total E: -475.715 eV
Step:    4 | Temp:  189.1 K | Epot: -478.174 eV | Total E: -475.730 eV
Step:    5 | Temp:  198.0 K | Epot: -478.305 eV | Total E: -475.745 eV
Step:    6 | Temp:  194.5 K | Epot: -478.256 eV | Total E: -475.743 eV
Step:    7 | Temp:  178.1 K | Epot: -478.030 eV | Total E: -475.728 eV
Step:    8 | Temp:  171.0 K | Epot: -477.916 eV | Total E: -475.706 eV
Step:    9 | Temp:  191.2 K | Epot: -478.180 eV | Total E: -475.708 eV
Step:   10 | Temp:  233.3 K | Epot: -478.744 eV | Total E: -475.729 eV
Step:   11 | Temp:  267.7 K | Epot: -479.221 eV | Total E: -475.761 eV
Step:   12 | Temp:  275.8 K | Epot: -479.314 eV | Total E: -475.749 eV
Step:   13 | Temp:  262.4 K | Epot: -479.117 eV | Total E: -475.724 eV
Step: 

Full analysis of the trajectory

In [ ]:
import os
import automatic_analysis5

input_traj_dir = "/content/drive/MyDrive/MDProject/results/md_trajectory_corr2_2k.extxyz"
output_dir = "/content/drive/MyDrive/MDProject/results/correlation/corr2_2k"

# Create the directory before running the analysis
os.makedirs(output_dir, exist_ok=True)

automatic_analysis5.full_analysis(input_traj_dir, output_dir, rmax=3.9, nbins=100, timestep_in_fs=0.5, window_size=2)

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************



[Total Energy] t0 = 835 (frame 835), g = 57.04, N_eff = 20.5 / 2001
N atoms (equilibrium):  100 in volume: 1474.4473736057512
Done computing for pair ('Sn', 'I')
Done computing for pair ('I', 'I')
Done computing for pair ('N', 'I')
Done computing for pair ('H', 'I')
Done computing for pair ('C', 'N')
Done computing for pair ('H', 'N')
Done computing for pair ('H', 'C')
Area (90° peak): 0.8112
Area (180° peak): 0.1888
Max v dataset: 0.14466144000000014
Processed 2001 frames successfully.
Processed 2001 frames successfully. Elements: [np.str_('C'), np.str_('H'), np.str_('I'), np.str_('N'), np.str_('Sn')]
Calculated mean temperature: 1245.61 K


## --correlation=3

In [ ]:
import os
import sys
import numpy as np
from ase.io import read, write
from google.colab import drive

from mace.cli.run_train import main

# Mount Google Drive to access the MDProject folder
drive.mount('/content/drive')

# Define file paths
outcar_file = "/content/drive/MyDrive/MDProject/OUTCAR"

data_dir = "/content/drive/MyDrive/MDProject/new_data/1.NVT_300/A.1_10K/corr3true"
my_sim_checkpoints_dir = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/corr3true"
my_sim_results_dir = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/results/corr3true"

os.makedirs(data_dir, exist_ok=True)

train_file = f"{data_dir}/train.extxyz"
valid_file = f"{data_dir}/valid.extxyz"
test_file = f"{data_dir}/test.extxyz"

# Start at frame 6840 and take every 10th frame
print("Reading and subsampling OUTCAR...")
frames = read(outcar_file, index="6840::10", format="vasp-out")
print(f"Extracted {len(frames)} distinct frames.")

# FIX ASE KEYS
print("Renaming ASE internal keys...")
for frame in frames:
    if 'energy' in frame.info:
        frame.info['REF_energy'] = frame.info.pop('energy')
    if 'stress' in frame.info:
        frame.info['REF_stress'] = frame.info.pop('stress')
    if 'forces' in frame.arrays:
        frame.arrays['REF_forces'] = frame.arrays.pop('forces')

# SPLIT: 80% Train, 10% Valid, 10% Test
n_total = len(frames)
n_train = int(0.8 * n_total)
n_valid = int(0.1 * n_total)

train_frames = frames[:n_train]
valid_frames = frames[n_train : n_train + n_valid]
test_frames = frames[n_train + n_valid:]

# SAVE
write(train_file, train_frames, format="extxyz")
write(valid_file, valid_frames, format="extxyz")
write(test_file, test_frames, format="extxyz")

print(f"Saved {len(train_frames)} train, {len(valid_frames)} valid, and {len(test_frames)} test frames.")

my_atomic_numbers = "[1, 6, 7, 50, 53]"

custom_args = [
    "mace_run_training",
    "--name=model_3000_pts_corr3",
    f"--train_file={train_file}",
    f"--valid_file={valid_file}",

    "--energy_key=energy",
    "--forces_key=forces",
    "--E0s=average",

    "--stress_weight=1.0",
    "--compute_stress=True",

    "--max_num_epochs=200",
    "--patience=10",
    "--batch_size=5",
    "--valid_batch_size=5",
    "--device=cuda",
    "--default_dtype=float32",

    "--r_max=5.0",
    "--correlation=3",

    f"--checkpoints_dir={my_sim_checkpoints_dir}",
    f"--results_dir={my_sim_results_dir}",
    "--keep_checkpoints",

    "--num_channels=64",
    "--max_L=0",
    "--seed=1",


    f"--atomic_numbers={my_atomic_numbers}",
    "--num_samples_pt=1000",
    "--lr=0.0001",

    "--energy_weight=1.0",
    "--forces_weight=100.0",

    "--optimizer=adamw",
    "--amsgrad",
    "--weight_decay=5e-7",

    "--ema",
    "--ema_decay=0.99",

    "--swa",
    "--start_swa=30",

    "--swa_forces_weight=100.0",
    "--swa_stress_weight=1.0",
    "--swa_energy_weight=1.0",

    "--scaling=rms_forces_scaling",
    "--restart_latest"
]

sys.argv = custom_args

if __name__ == "__main__":
    try:
        main()
        print("Training completed!")
    except Exception as e:
        print(f"Error: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading and subsampling OUTCAR...
Extracted 316 distinct frames.
Renaming ASE internal keys...


INFO:root:===========VERIFYING SETTINGS===========


Saved 252 train, 31 valid, and 33 test frames.
2026-09-21 12:49:36.033 INFO: ===========VERIFYING SETTINGS===========
2026-09-21 12:49:36.033 INFO: ===========VERIFYING SETTINGS===========
2026-09-21 12:49:36.033 INFO: ===========VERIFYING SETTINGS===========
2026-09-21 12:49:36.033 INFO: ===========VERIFYING SETTINGS===========


INFO:root:Stage Two is activated as start_stage_two was defined


2026-09-21 12:49:36.039 INFO: Stage Two is activated as start_stage_two was defined
2026-09-21 12:49:36.039 INFO: Stage Two is activated as start_stage_two was defined
2026-09-21 12:49:36.039 INFO: Stage Two is activated as start_stage_two was defined
2026-09-21 12:49:36.039 INFO: Stage Two is activated as start_stage_two was defined


...

2026-09-21 13:08:00.620 INFO: Evaluating valid_Default ...
2026-09-21 13:08:00.620 INFO: Evaluating valid_Default ...
2026-09-21 13:08:00.620 INFO: Evaluating valid_Default ...
2026-09-21 13:08:00.620 INFO: Evaluating valid_Default ...


INFO:root:Error-table on TRAIN and VALID:
+---------------+---------------------+------------------+-------------------+
|  config_type  | RMSE E / meV / atom | RMSE F / meV / A | relative F RMSE % |
+---------------+---------------------+------------------+-------------------+
| train_Default |            1.9      |         46.2     |         13.75     |
| valid_Default |            1.6      |         56.1     |         16.20     |
+---------------+---------------------+------------------+-------------------+


2026-09-21 13:08:00.916 INFO: Error-table on TRAIN and VALID:
+---------------+---------------------+------------------+-------------------+
|  config_type  | RMSE E / meV / atom | RMSE F / meV / A | relative F RMSE % |
+---------------+---------------------+------------------+-------------------+
| train_Default |            1.9      |         46.2     |         13.75     |
| valid_Default |            1.6      |         56.1     |         16.20     |
+---------------+---------------------+------------------+-------------------+
2026-09-21 13:08:00.916 INFO: Error-table on TRAIN and VALID:
+---------------+---------------------+------------------+-------------------+
|  config_type  | RMSE E / meV / atom | RMSE F / meV / A | relative F RMSE % |
+---------------+---------------------+------------------+-------------------+
| train_Default |            1.9      |         46.2     |         13.75     |
| valid_Default |            1.6      |         56.1     |         16.20     |
+------

DEBUG:root:Running inference on train_Default dataset
DEBUG:root:Running inference on valid_Default dataset
INFO:root:Done


2026-09-21 13:08:06.633 INFO: Done
2026-09-21 13:08:06.633 INFO: Done
2026-09-21 13:08:06.633 INFO: Done
2026-09-21 13:08:06.633 INFO: Done
Training completed!


Simulating trajectories with ase

In [ ]:
import os
import time
from ase.io import read, write
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from mace.calculators import MACECalculator
from google.colab import drive

drive.mount('/content/drive')

model_path = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/corr3true/model_3000_pts_corr3_run-1.model"
init_structure_path = "/content/drive/MyDrive/MDProject/new_data/1.NVT_300/A.1_10K/Smart_FPS/valid_fps.extxyz"
output_traj_path = "/content/drive/MyDrive/MDProject/results/md_trajectory_corr3_2k.extxyz"


atoms = read(init_structure_path, index=0)

calculator = MACECalculator(
    model_paths=[model_path],
    device='cuda',
    default_dtype='float32'
)
atoms.calc = calculator

temperature_K = 300
timestep_fs = 1.0
friction_coeff = 0.001

print(f"Initializing velocities at {temperature_K} K...")
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)

dyn = Langevin(
    atoms,
    timestep_fs * units.fs,
    temperature_K=temperature_K,
    friction=friction_coeff
)

#  Define an output function to save frames
def write_frame():
    epot = atoms.get_potential_energy()
    ekin = atoms.get_kinetic_energy()
    temp = ekin / (1.5 * units.kB * len(atoms))

    # Append to the .extxyz trajectory file
    write(output_traj_path, atoms, append=True)

    # Print status every 10 steps
    step = dyn.get_number_of_steps()
    print(f"Step: {step:>4} | Temp: {temp:>6.1f} K | Epot: {epot:+.3f} eV | Total E: {epot+ekin:+.3f} eV")

# Clear old trajectory file if it exists
if os.path.exists(output_traj_path):
    os.remove(output_traj_path)

# Attach the logger to run every 10 steps
dyn.attach(write_frame, interval=1)

#  Run the Simulation
num_steps = 2000
print(f"\nStarting MD simulation for {num_steps} steps...")
start_time = time.time()

dyn.run(num_steps)

print(f"\nMD simulation finished in {time.time() - start_time:.2f} seconds!")
print(f"Trajectory safely saved to: {output_traj_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading initial structure...
Loading compiled MACE model from: /content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/corr3true/model_3000_pts_corr3_run-1.model...


/usr/local/lib/python3.13/dist-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Initializing velocities at 300 K...

Starting MD simulation for 2000 steps...
Step:    0 | Temp:  340.9 K | Epot: -510.805 eV | Total E: -506.398 eV


/tmp/ipykernel_565/2075710288.py:38: DeprecationWarning: Use thermalize_momenta
  MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)
/usr/local/lib/python3.13/dist-packages/ase/md/langevin.py:102: FutureWarning: The implementation of `fixcm=True` in `Langevin` does not strictly sample the correct NVT distributions. The deviations are typically small for large systems but can be more pronounced for small systems. Use `fixcm=False` together with `ase.constraints.FixCom`. `fixcm` is deprecated since ASE 3.28.0 and will be removed in a future release.
  warnings.warn(msg, FutureWarning)


Step:    1 | Temp:  323.0 K | Epot: -510.548 eV | Total E: -506.373 eV
Step:    2 | Temp:  281.8 K | Epot: -509.996 eV | Total E: -506.354 eV
Step:    3 | Temp:  249.5 K | Epot: -509.572 eV | Total E: -506.347 eV
Step:    4 | Temp:  242.4 K | Epot: -509.487 eV | Total E: -506.353 eV
Step:    5 | Temp:  250.0 K | Epot: -509.610 eV | Total E: -506.379 eV
Step:    6 | Temp:  248.6 K | Epot: -509.599 eV | Total E: -506.386 eV
Step:    7 | Temp:  224.9 K | Epot: -509.281 eV | Total E: -506.374 eV
Step:    8 | Temp:  194.9 K | Epot: -508.866 eV | Total E: -506.347 eV
Step:    9 | Temp:  182.5 K | Epot: -508.696 eV | Total E: -506.337 eV
Step:   10 | Temp:  197.1 K | Epot: -508.902 eV | Total E: -506.354 eV
Step:   11 | Temp:  224.8 K | Epot: -509.283 eV | Total E: -506.377 eV
Step:   12 | Temp:  240.1 K | Epot: -509.496 eV | Total E: -506.392 eV
Step:   13 | Temp:  237.1 K | Epot: -509.448 eV | Total E: -506.383 eV
Step:   14 | Temp:  231.2 K | Epot: -509.358 eV | Total E: -506.369 eV
Step: 

In [ ]:
import os
import automatic_analysis5

input_traj_dir = "/content/drive/MyDrive/MDProject/results/md_trajectory_corr3_2k.extxyz"
output_dir = "/content/drive/MyDrive/MDProject/results/correlation/corr3_2k"

# Create the directory before running the analysis
os.makedirs(output_dir, exist_ok=True)

automatic_analysis5.full_analysis(input_traj_dir, output_dir, rmax=3.9, nbins=100, timestep_in_fs=0.5, window_size=2)

[Total Energy] t0 = 1282 (frame 1282), g = 9.53, N_eff = 75.5 / 2001
N atoms (equilibrium):  100 in volume: 1474.4473736057512
Done computing for pair ('Sn', 'I')
Done computing for pair ('I', 'I')
Done computing for pair ('N', 'I')
Done computing for pair ('H', 'I')
Done computing for pair ('C', 'N')
Done computing for pair ('H', 'N')
Done computing for pair ('H', 'C')
Area (90° peak): 0.9825
Area (180° peak): 0.0175
Max v dataset: 0.13247991999999975
Processed 2001 frames successfully.
Processed 2001 frames successfully. Elements: [np.str_('C'), np.str_('H'), np.str_('I'), np.str_('N'), np.str_('Sn')]
Calculated mean temperature: 1134.46 K


## --correlation=4

In [ ]:
import os
import sys
import numpy as np
from ase.io import read, write
from google.colab import drive

from mace.cli.run_train import main

# Mount Google Drive to access the MDProject folder
drive.mount('/content/drive')

# Define file paths
outcar_file = "/content/drive/MyDrive/MDProject/OUTCAR"

data_dir = "/content/drive/MyDrive/MDProject/new_data/1.NVT_300/A.1_10K/corr4true"
my_sim_checkpoints_dir = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/corr4true"
my_sim_results_dir = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/results/corr4true"

os.makedirs(data_dir, exist_ok=True)

train_file = f"{data_dir}/train.extxyz"
valid_file = f"{data_dir}/valid.extxyz"
test_file = f"{data_dir}/test.extxyz"

# Start at frame 6840 and take every 10th frame
print("Reading and subsampling OUTCAR...")
frames = read(outcar_file, index="6840::10", format="vasp-out")
print(f"Extracted {len(frames)} distinct frames.")

# FIX ASE KEYS
print("Renaming ASE internal keys...")
for frame in frames:
    if 'energy' in frame.info:
        frame.info['REF_energy'] = frame.info.pop('energy')
    if 'stress' in frame.info:
        frame.info['REF_stress'] = frame.info.pop('stress')
    if 'forces' in frame.arrays:
        frame.arrays['REF_forces'] = frame.arrays.pop('forces')

# SPLIT: 80% Train, 10% Valid, 10% Test
n_total = len(frames)
n_train = int(0.8 * n_total)
n_valid = int(0.1 * n_total)

train_frames = frames[:n_train]
valid_frames = frames[n_train : n_train + n_valid]
test_frames = frames[n_train + n_valid:]

# SAVE
write(train_file, train_frames, format="extxyz")
write(valid_file, valid_frames, format="extxyz")
write(test_file, test_frames, format="extxyz")

print(f"Saved {len(train_frames)} train, {len(valid_frames)} valid, and {len(test_frames)} test frames.")

my_atomic_numbers = "[1, 6, 7, 50, 53]"

custom_args = [
    "mace_run_training",
    "--name=model_3000_pts_corr4",
    f"--train_file={train_file}",
    f"--valid_file={valid_file}",

    "--energy_key=energy",
    "--forces_key=forces",
    "--E0s=average",

    "--stress_weight=1.0",
    "--compute_stress=True",

    "--max_num_epochs=200",
    "--patience=10",
    "--batch_size=5",
    "--valid_batch_size=5",
    "--device=cuda",
    "--default_dtype=float32",

    "--r_max=5.0",
    "--correlation=4",

    f"--checkpoints_dir={my_sim_checkpoints_dir}",
    f"--results_dir={my_sim_results_dir}",
    "--keep_checkpoints",

    "--num_channels=64",
    "--max_L=0",
    "--seed=1",


    f"--atomic_numbers={my_atomic_numbers}",
    "--num_samples_pt=1000",
    "--lr=0.0001",

    "--energy_weight=1.0",
    "--forces_weight=100.0",

    "--optimizer=adamw",
    "--amsgrad",
    "--weight_decay=5e-7",

    "--ema",
    "--ema_decay=0.99",

    "--swa",
    "--start_swa=30",

    "--swa_forces_weight=100.0",
    "--swa_stress_weight=1.0",
    "--swa_energy_weight=1.0",

    "--scaling=rms_forces_scaling",
    "--restart_latest"
]

sys.argv = custom_args

if __name__ == "__main__":
    try:
        main()
        print("Training completed!")
    except Exception as e:
        print(f"Error: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading and subsampling OUTCAR...
Extracted 316 distinct frames.
Renaming ASE internal keys...


INFO:root:===========VERIFYING SETTINGS===========


Saved 252 train, 31 valid, and 33 test frames.
2026-09-22 15:04:42.638 INFO: ===========VERIFYING SETTINGS===========


INFO:root:Stage Two is activated as start_stage_two was defined


2026-09-22 15:04:42.641 INFO: Stage Two is activated as start_stage_two was defined


...

2026-09-22 15:07:54.035 INFO: Error-table on TRAIN and VALID:
+---------------+---------------------+------------------+-------------------+
|  config_type  | RMSE E / meV / atom | RMSE F / meV / A | relative F RMSE % |
+---------------+---------------------+------------------+-------------------+
| train_Default |          321.4      |        224.1     |         66.69     |
| valid_Default |          315.4      |        226.3     |         65.33     |
+---------------+---------------------+------------------+-------------------+


DEBUG:root:Running inference on train_Default dataset
DEBUG:root:Running inference on valid_Default dataset
INFO:root:Loading checkpoint: /content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/corr4true/model_3000_pts_corr4_run-1_epoch-198_swa.pt


2026-09-22 15:08:12.079 INFO: Loading checkpoint: /content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/corr4true/model_3000_pts_corr4_run-1_epoch-198_swa.pt


/usr/local/lib/python3.13/dist-packages/mace/tools/checkpoint.py:187: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=checkpoint_info.path, map_location=device),


Error: [Errno 103] Software caused connection abort


Simulating trajectories with ase

In [ ]:
import os
import time
from ase.io import read, write
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from mace.calculators import MACECalculator
from google.colab import drive

#  Mount Google Drive
drive.mount('/content/drive')

model_path ="/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/corr4true/model_3000_pts_corr4_run-1.model"
init_structure_path = "/content/drive/MyDrive/MDProject/new_data/1.NVT_300/A.1_10K/Smart_FPS/valid_fps.extxyz"
output_traj_path = "/content/drive/MyDrive/MDProject/results/md_trajectory_corr4_2k.extxyz"

atoms = read(init_structure_path, index=0)


calculator = MACECalculator(
    model_paths=[model_path],
    device='cuda',
    default_dtype='float32'
)
atoms.calc = calculator


temperature_K = 300
timestep_fs = 1.0
friction_coeff = 0.001

print(f"Initializing velocities at {temperature_K} K...")
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)

dyn = Langevin(
    atoms,
    timestep_fs * units.fs,
    temperature_K=temperature_K,
    friction=friction_coeff
)

#  Define an output function to save frames
def write_frame():
    epot = atoms.get_potential_energy()
    ekin = atoms.get_kinetic_energy()
    temp = ekin / (1.5 * units.kB * len(atoms))

    # Append to the .extxyz trajectory file
    write(output_traj_path, atoms, append=True)

    # Print status every 10 steps
    step = dyn.get_number_of_steps()
    print(f"Step: {step:>4} | Temp: {temp:>6.1f} K | Epot: {epot:+.3f} eV | Total E: {epot+ekin:+.3f} eV")

# Clear old trajectory file if it exists
if os.path.exists(output_traj_path):
    os.remove(output_traj_path)

# Attach the logger to run every 10 steps
dyn.attach(write_frame, interval=1)

#  Run the Simulation
num_steps = 2000
print(f"\nStarting MD simulation for {num_steps} steps...")
start_time = time.time()

dyn.run(num_steps)

print(f"\nMD simulation finished in {time.time() - start_time:.2f} seconds!")
print(f"Trajectory safely saved to: {output_traj_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading initial structure...
Loading compiled MACE model from: /content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/corr4true/model_3000_pts_corr4_run-1.model...


/usr/local/lib/python3.13/dist-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Initializing velocities at 300 K...

Starting MD simulation for 2000 steps...
Step:    0 | Temp:  340.0 K | Epot: -511.053 eV | Total E: -506.658 eV


/tmp/ipykernel_565/4095474794.py:38: DeprecationWarning: Use thermalize_momenta
  MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)
/usr/local/lib/python3.13/dist-packages/ase/md/langevin.py:102: FutureWarning: The implementation of `fixcm=True` in `Langevin` does not strictly sample the correct NVT distributions. The deviations are typically small for large systems but can be more pronounced for small systems. Use `fixcm=False` together with `ase.constraints.FixCom`. `fixcm` is deprecated since ASE 3.28.0 and will be removed in a future release.
  warnings.warn(msg, FutureWarning)


Step:    1 | Temp:  325.6 K | Epot: -510.855 eV | Total E: -506.646 eV
Step:    2 | Temp:  292.2 K | Epot: -510.403 eV | Total E: -506.627 eV
Step:    3 | Temp:  264.4 K | Epot: -510.032 eV | Total E: -506.615 eV
Step:    4 | Temp:  255.3 K | Epot: -509.917 eV | Total E: -506.618 eV
Step:    5 | Temp:  258.5 K | Epot: -509.971 eV | Total E: -506.630 eV
Step:    6 | Temp:  258.3 K | Epot: -509.982 eV | Total E: -506.644 eV
Step:    7 | Temp:  247.9 K | Epot: -509.842 eV | Total E: -506.639 eV
Step:    8 | Temp:  232.0 K | Epot: -509.632 eV | Total E: -506.633 eV
Step:    9 | Temp:  223.4 K | Epot: -509.512 eV | Total E: -506.624 eV
Step:   10 | Temp:  225.6 K | Epot: -509.547 eV | Total E: -506.631 eV
Step:   11 | Temp:  233.4 K | Epot: -509.650 eV | Total E: -506.633 eV
Step:   12 | Temp:  236.4 K | Epot: -509.682 eV | Total E: -506.627 eV
Step:   13 | Temp:  235.0 K | Epot: -509.647 eV | Total E: -506.609 eV
Step:   14 | Temp:  238.4 K | Epot: -509.692 eV | Total E: -506.610 eV
Step: 

In [ ]:
import os
import automatic_analysis5

input_traj_dir = "/content/drive/MyDrive/MDProject/results/md_trajectory_corr4_2k.extxyz"
output_dir = "/content/drive/MyDrive/MDProject/results/correlation/corr4_2k"

# Create the directory before running the analysis
os.makedirs(output_dir, exist_ok=True)

automatic_analysis5.full_analysis(input_traj_dir, output_dir, rmax=3.9, nbins=100, timestep_in_fs=0.5, window_size=2)

[Total Energy] t0 = 1714 (frame 1714), g = 2.52, N_eff = 114.3 / 2001
N atoms (equilibrium):  100 in volume: 1474.4473736057512
Done computing for pair ('Sn', 'I')
Done computing for pair ('I', 'I')
Done computing for pair ('N', 'I')
Done computing for pair ('H', 'I')
Done computing for pair ('C', 'N')
Done computing for pair ('H', 'N')
Done computing for pair ('H', 'C')
Area (90° peak): 0.7525
Area (180° peak): 0.2475
Max v dataset: 2.08360248
Processed 2001 frames successfully.
Processed 2001 frames successfully. Elements: [np.str_('C'), np.str_('H'), np.str_('I'), np.str_('N'), np.str_('Sn')]
Calculated mean temperature: 16345.92 K


In [ ]:
! pip install ase
! pip install pymbar
! pip install mace-torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 9.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.7/387.7 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.8/222.8 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.1/453.1 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 7.4 MB/s eta 0:00:00
  Created wheel for python-hostlist: filename=python_hostlist-2.3.0-py3-none-any.whl size=39523 sha256=6888f57a52d7ffc39b20c7717b32e20eb4b09a6a65c87a987c6fc9cb42eac05e
  Stored in directory: /root/.cache/pip/wheels/05/2e/7e/39f980a5a0755f9